# Preprocess Pipeline and Classical Feature Extraction

In [9]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import os
import numpy as np
from src.config import DATABASE_CONFIGS
from src.utils import load_emg_data
from src.preprocess import (
    apply_bandpass_filter,
    sliding_window_with_labels,
    split_by_repetition
)
from src.features import extract_features_batch

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## CONFIGURATION

In [10]:
DATABASE = 'DB5'
SUBJECT = 1
ZIP_PATH = f'../data/raw/Ninapro_{DATABASE}/s{SUBJECT}.zip'
SAVE_DIR = f'../data/preprocessed/{DATABASE}/'

os.makedirs(SAVE_DIR, exist_ok=True)
config = DATABASE_CONFIGS[DATABASE]

print(f"Database: {DATABASE} — {config['description']}")
print(f"Subject: {SUBJECT}")
print(f"Sampling rate: {config['fs']} Hz")
print(f"Channels: {config['n_channels']}")
print(f"Window: {config['window_ms']}ms with {config['overlap_pct']*100}% overlap")
print(f"Test repetitions: {config['test_reps']}")

Database: DB5 — 52 gestures, 16 Myo armband, 200 Hz
Subject: 1
Sampling rate: 200 Hz
Channels: 16
Window: 200ms with 50.0% overlap
Test repetitions: [2, 5]


## 1. Load raw data

In [11]:
print("\n[Step 1/6] Loading raw data...")
df_emg, stimulus, restimulus, repetition = load_emg_data(ZIP_PATH)
print(f"  EMG shape: {df_emg.shape}")
print(f"  Label range: {restimulus.min()} to {restimulus.max()}")


[Step 1/6] Loading raw data...
  EMG shape: (568540, 16)
  Label range: 0 to 23


## 2. Band-Pass Filtering

In [12]:
print(f"\n[Step 2/6] Bandpass filtering ({config['lowcut']}-{config['highcut']} Hz)...")
df_filtered = apply_bandpass_filter(
    df_emg.values,
    lowcut=config['lowcut'],
    highcut=config['highcut'],
    fs=config['fs'],
    order=4
)
print(f"  Filtered shape: {df_filtered.shape}")


[Step 2/6] Bandpass filtering (15.0-90.0 Hz)...
  Filtered shape: (568540, 16)


## 3. Sliding Window

In [13]:
print(f"\n[Step 3/6] Sliding window ({config['window_ms']}ms, {config['overlap_pct']*100}% overlap)...")
X_windows, y_windows, rep_windows = sliding_window_with_labels(
    df_filtered,
    restimulus,  # Use restimulus NOT stimulus — correctly marks rest periods
    repetition,
    window_ms=config['window_ms'],
    overlap_pct=config['overlap_pct'],
    fs=config['fs']
)

window_samples = int(config['window_ms'] / 1000 * config['fs'])
print(f"  Windows shape: {X_windows.shape} → {X_windows.shape[0]} windows of {window_samples} samples × {config['n_channels']} channels")
print(f"  Signal dimension per window: {window_samples * config['n_channels']}")


[Step 3/6] Sliding window (200ms, 50.0% overlap)...
  Windows shape: (28426, 40, 16) → 28426 windows of 40 samples × 16 channels
  Signal dimension per window: 640


## 4. Split by repetition

In [14]:
print(f"\n[Step 4/6] Splitting by repetition (test reps: {config['test_reps']})...")
X_train, X_test, y_train, y_test = split_by_repetition(
    X_windows, y_windows, rep_windows, test_reps=config['test_reps']
)


[Step 4/6] Splitting by repetition (test reps: [2, 5])...
Train: 7826 windows from reps [np.int32(0), np.int32(1), np.int32(3), np.int32(4), np.int32(6)]
Test:  3857 windows from reps [np.int32(2), np.int32(5)]
Train classes: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]
Test classes:  [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


## 5. Extract classical features for baseline comparison

In [15]:
print(f"\n[Step 5/6] Extracting classical time-domain features...")
X_train_classical = extract_features_batch(X_train)
X_test_classical = extract_features_batch(X_test)
print(f"  Train features: {X_train_classical.shape}")
print(f"  Test features:  {X_test_classical.shape}")
print(f"  Feature dimension: {X_train_classical.shape[1]} ({config['n_channels']} channels × 5 features)")


[Step 5/6] Extracting classical time-domain features...
  Train features: (7826, 80)
  Test features:  (3857, 80)
  Feature dimension: 80 (16 channels × 5 features)


## 6. Save

In [16]:
print(f"\n[Step 6/6] Saving preprocessed data to {SAVE_DIR}...")

prefix = f'S{SUBJECT:02d}'

np.save(os.path.join(SAVE_DIR, f'{prefix}_X_windows.npy'), X_windows)
np.save(os.path.join(SAVE_DIR, f'{prefix}_y_windows.npy'), y_windows)
np.save(os.path.join(SAVE_DIR, f'{prefix}_rep_windows.npy'), rep_windows)

np.save(os.path.join(SAVE_DIR, f'{prefix}_X_train.npy'), X_train)
np.save(os.path.join(SAVE_DIR, f'{prefix}_X_test.npy'), X_test)
np.save(os.path.join(SAVE_DIR, f'{prefix}_y_train.npy'), y_train)
np.save(os.path.join(SAVE_DIR, f'{prefix}_y_test.npy'), y_test)

np.save(os.path.join(SAVE_DIR, f'{prefix}_X_train_classical.npy'), X_train_classical)
np.save(os.path.join(SAVE_DIR, f'{prefix}_X_test_classical.npy'), X_test_classical)

print(f"\nFiles saved:")
for f in sorted(os.listdir(SAVE_DIR)):
    if f.startswith(prefix):
        file_path = os.path.join(SAVE_DIR, f)
        size_kb = os.path.getsize(file_path) / 1024
        print(f"  {f:<40} {size_kb:>8.1f} KB")

print(f"\nPreprocessing complete!")



[Step 6/6] Saving preprocessed data to ../data/preprocessed/DB5/...

Files saved:
  S01_D_ksvd_N80_k10.npy                       25.1 KB
  S01_D_ksvd_N80_k5.npy                        25.1 KB
  S01_D_mb_N80_k10.npy                         12.6 KB
  S01_D_mb_N80_k5.npy                          12.6 KB
  S01_EMG.npy                               71062.6 KB
  S01_EMG_test.npy                           4815.1 KB
  S01_EMG_train.npy                         19527.6 KB
  S01_EMG_val.npy                            4862.6 KB
  S01_LABELS.npy                              222.2 KB
  S01_LABELS_test.npy                          15.2 KB
  S01_LABELS_train.npy                         61.1 KB
  S01_LABELS_val.npy                           15.3 KB
  S01_REPS.npy                                222.2 KB
  S01_X_features_classical.npy              17766.4 KB
  S01_X_test.npy                             9642.6 KB
  S01_X_test_classical.npy                   2410.8 KB
  S01_X_test_sparse.npy              